> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 07 · AGENTIC AI</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Tools, state, checkpointing, evals, guardrails</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">Multi-step workflows · recovery · observability · the eval harness that wins pilots</div>
</div>

**Time:** 75 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹8 &nbsp;·&nbsp; **Prereq:** Labs 05, 06

In [1]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path

# pip install sarvamai python-dotenv

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [2]:
# ── The ₹ meter, imported ─────────────────────────────────────────────────
# Lab 00 writes cost_meter.py next to these notebooks. If this import fails,
# run Lab 00 once — it is the only lab that defines the meter.
try:
    from cost_meter import CostMeter, RATES, FREE_CREDIT
except ImportError:
    raise ImportError(
        "cost_meter.py not found.\n"
        "Run 00_Setup_and_the_Cost_Meter.ipynb once — its last section writes "
        "cost_meter.py into this folder, and every other lab imports it from there."
    )

cost = CostMeter()
print(f"cost meter armed · rates dated Aug 2026 · ₹{FREE_CREDIT:.0f} free credit")


cost meter armed · rates dated Aug 2026 · ₹1000 free credit


## Chatbot vs agent

| | Chatbot | Agent |
|---|---|---|
| Does | Answers a question | Completes a task |
| State | None, or yours | Explicit, across many steps |
| Acts | No | **Yes — writes to systems** |
| Failure | A bad answer | A wrong action, possibly irreversible |
| Evaluate on | Response quality | Task success, tool accuracy, recovery |

The moment an LLM can take an action, the engineering problem stops being prompt
quality and becomes **reliability, observability and blast radius**.

---
## 1 · The design law: LLMs reason, code executes

Push everything deterministic *out* of the model. It is cheaper, auditable, and correct.

In [3]:
# ❌ The expensive way — ask the model to compute
r = client.chat.completions(
    model="sarvam-105b", max_tokens=2500,
    messages=[{"role": "user", "content":
               "Loan ₹500000, 9.5% annual reducing, 60 months. Compute the exact EMI."}])
cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
print("LLM says:", (r.choices[0].message.content or "")[-200:])

LLM says: 


In [4]:
# ✅ The right way — model decides WHICH calculation, code performs it
def emi(principal: float, annual_rate: float, months: int) -> dict:
    r = annual_rate / 12 / 100
    m = principal * r * (1 + r) ** months / ((1 + r) ** months - 1)
    return {"emi": round(m, 2), "total_paid": round(m * months, 2),
            "total_interest": round(m * months - principal, 2)}

print("code says:", emi(500000, 9.5, 60))

code says: {'emi': 10500.93, 'total_paid': 630055.84, 'total_interest': 130055.84}


> Deterministic, free at the point of execution, unit-testable, and identical every
> time. Arya reports **114% better performance on complex tasks** using smaller, cheaper
> models by applying exactly this principle.

---
## 2 · A stateful, checkpointed agent

A failure at step 40 must not lose the first 39.

In [5]:
import sqlite3, uuid
from dataclasses import dataclass, field, asdict

DB = sqlite3.connect(":memory:")
DB.execute("CREATE TABLE ckpt (run_id TEXT, step INT, state TEXT, ts REAL)")

@dataclass
class RunState:
    run_id: str
    step: int = 0
    data: dict = field(default_factory=dict)
    trace: list = field(default_factory=list)

def save(st: RunState):
    DB.execute("INSERT INTO ckpt VALUES (?,?,?,?)",
               (st.run_id, st.step, json.dumps(asdict(st)), time.time())); DB.commit()

def restore(run_id: str):
    row = DB.execute("SELECT state FROM ckpt WHERE run_id=? ORDER BY step DESC LIMIT 1",
                     (run_id,)).fetchone()
    return RunState(**json.loads(row[0])) if row else None

print("checkpoint store ready")

checkpoint store ready


In [6]:
# ── A 5-step loan triage workflow ─────────────────────────────────────────
def step_extract(st):
    st.data["applicant"] = {"name": "Rajesh Kumar", "income": 65000,
                            "pan": "ABCDE1234F", "loan_amount": 500000}
    return "extracted applicant fields"

def step_validate(st):
    a = st.data["applicant"]
    errs = []
    if len(a["pan"]) != 10: errs.append("PAN malformed")
    if a["income"] <= 0:    errs.append("income missing")
    st.data["validation"] = {"ok": not errs, "errors": errs}
    return f"validation ok={not errs}"

def step_eligibility(st):
    a = st.data["applicant"]
    e = emi(a["loan_amount"], 9.5, 60)
    ratio = e["emi"] / a["income"]
    st.data["eligibility"] = {"emi": e["emi"], "foir": round(ratio, 3),
                              "eligible": ratio < 0.5}
    return f"FOIR={ratio:.2f} eligible={ratio < 0.5}"

def step_risk(st):
    if st.data.get("_inject_failure"):
        raise RuntimeError("credit bureau timeout")
    st.data["risk"] = {"bureau_score": 748, "band": "low"}
    return "bureau score 748"

def step_decide(st):
    ok = st.data["eligibility"]["eligible"] and st.data["risk"]["band"] == "low"
    st.data["decision"] = "APPROVE" if ok else "ESCALATE"
    return st.data["decision"]

STEPS = [step_extract, step_validate, step_eligibility, step_risk, step_decide]

In [7]:
def run_workflow(run_id=None, resume=False, inject_failure=False):
    st = restore(run_id) if resume else RunState(run_id=run_id or str(uuid.uuid4())[:8])
    st.data["_inject_failure"] = inject_failure
    print(f"{'RESUMING' if resume else 'STARTING'} run {st.run_id} at step {st.step}")

    while st.step < len(STEPS):
        fn = STEPS[st.step]
        try:
            t0 = time.perf_counter()
            msg = fn(st)
            st.trace.append({"step": st.step, "fn": fn.__name__, "ok": True,
                             "ms": round((time.perf_counter()-t0)*1000, 1), "msg": msg})
            print(f"  ✅ {st.step} {fn.__name__:<18} {msg}")
            st.step += 1
            save(st)                       # ← checkpoint AFTER each success
        except Exception as e:
            st.trace.append({"step": st.step, "fn": fn.__name__, "ok": False, "err": str(e)})
            print(f"  ❌ {st.step} {fn.__name__:<18} {e}")
            save(st)
            return st, False
    return st, True

st, ok = run_workflow(run_id="demo1", inject_failure=True)
print("\ncompleted:", ok, "| stopped at step", st.step)

STARTING run demo1 at step 0
  ✅ 0 step_extract       extracted applicant fields
  ✅ 1 step_validate      validation ok=True
  ✅ 2 step_eligibility   FOIR=0.16 eligible=True
  ❌ 3 step_risk          credit bureau timeout

completed: False | stopped at step 3


In [8]:
# Resume from the checkpoint — steps 0-2 are NOT re-run
st2, ok2 = run_workflow(run_id="demo1", resume=True, inject_failure=False)
print("\ncompleted:", ok2, "| decision:", st2.data.get("decision"))
print("\nFull trace:")
for t in st2.trace: print("  ", t)

RESUMING run demo1 at step 3
  ✅ 3 step_risk          bureau score 748
  ✅ 4 step_decide        APPROVE

completed: True | decision: APPROVE

Full trace:
   {'step': 0, 'fn': 'step_extract', 'ok': True, 'ms': 0.0, 'msg': 'extracted applicant fields'}
   {'step': 1, 'fn': 'step_validate', 'ok': True, 'ms': 0.0, 'msg': 'validation ok=True'}
   {'step': 2, 'fn': 'step_eligibility', 'ok': True, 'ms': 0.0, 'msg': 'FOIR=0.16 eligible=True'}
   {'step': 3, 'fn': 'step_risk', 'ok': True, 'ms': 0.0, 'msg': 'bureau score 748'}
   {'step': 4, 'fn': 'step_decide', 'ok': True, 'ms': 0.0, 'msg': 'APPROVE'}


> **This is the single biggest reliability difference at scale.** A workflow that must
> restart from step 1 every time something transient fails cannot run a fifty-step
> compliance review. It can only demo.

---
## 3 · Observability — you cannot debug what you cannot see

This block gives every step of a run a **latency, token count and ₹ cost**, collected
into one report — the minimum you need to answer "why is this run slow/expensive"
without re-running it under a debugger.

It's two classes because it splits **collection** from **measurement**:

- `Tracer` owns the results — a flat list of finished spans, plus `.report()` to
  print them as a table with totals. One `Tracer` per run.
- `_Span` owns one measurement's lifecycle, scoped to a `with` block. It is a
  **context manager**: `__enter__` starts the clock, `__exit__` stops it and pushes
  the finished record onto its parent `Tracer`'s list. `.set()` lets you attach
  extra metadata (tokens, cost) from inside the block before it closes.

`with tracer.span("classify") as sp:` reads naturally because that's exactly what
happens — `tracer.span()` hands you a fresh `_Span` wired back to `tracer`, so
entering/leaving the block automatically times the work and records it, with no
manual `tracer.spans.append(...)` bookkeeping. The leading underscore on `_Span`
signals it's not meant to be built directly — always go through `tracer.span(...)`.

In [9]:
class Tracer:
    def __init__(self): self.spans = []
    def span(self, name, **meta):
        return _Span(self, name, meta)
    def report(self):
        print(f"{'span':<26}{'ms':>9}{'tokens':>9}{'₹':>10}")
        print("─" * 54)
        for s in self.spans:
            print(f"{s['name']:<26}{s['ms']:>9.0f}{s.get('tokens',0):>9}{s.get('inr',0):>10.4f}")
        print("─" * 54)
        print(f"{'TOTAL':<26}{sum(s['ms'] for s in self.spans):>9.0f}"
              f"{sum(s.get('tokens',0) for s in self.spans):>9}"
              f"{sum(s.get('inr',0) for s in self.spans):>10.4f}")

class _Span:
    def __init__(self, tr, name, meta): self.tr, self.name, self.meta = tr, name, meta
    def __enter__(self): self.t0 = time.perf_counter(); return self
    def __exit__(self, *a):
        self.tr.spans.append({"name": self.name, "ms": (time.perf_counter()-self.t0)*1000,
                              **self.meta})
    def set(self, **kw): self.meta.update(kw)

tracer = Tracer()

with tracer.span("classify") as sp:
    r = client.chat.completions(model="sarvam-105b", max_tokens=300, reasoning_effort=None,
        messages=[{"role":"user","content":"Classify: 'my EMI bounced twice'"}])
    inr = cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
    sp.set(tokens=r.usage.total_tokens, inr=inr)

with tracer.span("emi_calc") as sp:
    emi(500000, 9.5, 60)

tracer.report()

span                             ms   tokens         ₹
──────────────────────────────────────────────────────
classify                       4702      282    0.0197
emi_calc                          0        0    0.0000
──────────────────────────────────────────────────────
TOTAL                          4702      282    0.0197


---
## 4 · Guardrails — PII redaction before you log anything

In [10]:
# Redaction - Hiding the sensitive personal informations in a text

import re

PII = [
    (re.compile(r"\b[A-Z]{5}[0-9]{4}[A-Z]\b"),                "[PAN]"),
    (re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b"),              "[AADHAAR]"),
    (re.compile(r"\b(?:\+91[- ]?)?[6-9]\d{9}\b"),            "[PHONE]"),
    (re.compile(r"\b\d{9,18}\b"),                            "[ACCOUNT]"),
    (re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),                  "[EMAIL]"),
]

def redact(text: str) -> str:
    for pat, tag in PII:
        text = pat.sub(tag, text)
    return text

SAMPLE1 = ("Rajesh, PAN ABCDE1234F, Aadhaar 1234 5678 9012,  "
          "a/c 123456789013, phone +91 9876543210, email rajesh@example.com — EMI bounced.")
SAMPLE2 = ("My name is Rajesh and  PAN number is ABCDE1234F. My Aadhaar id is 1234 5678 9012,  "
          "my a/c number is 123456789013, my phone +91 9876543210 and email rajesh@example.com — EMI bounced.")
SAMPLE3 = ("My required details are as under: Rajesh ABCDE1234F  1234 5678 9012,  "
          " 123456789013,  +91 9876543210,  rajesh@example.com — EMI bounced.")
# print("RAW     :", SAMPLE)
print("REDACTED:", redact(SAMPLE1))
print("REDACTED:", redact(SAMPLE2))
print("REDACTED:", redact(SAMPLE3))

REDACTED: Rajesh, PAN [PAN], Aadhaar [AADHAAR],  a/c [AADHAAR], phone +91 [PHONE], email [EMAIL] — EMI bounced.
REDACTED: My name is Rajesh and  PAN number is [PAN]. My Aadhaar id is [AADHAAR],  my a/c number is [AADHAAR], my phone +91 [PHONE] and email [EMAIL] — EMI bounced.
REDACTED: My required details are as under: Rajesh [PAN]  [AADHAAR],   [AADHAAR],  +91 [PHONE],  [EMAIL] — EMI bounced.


In [11]:
# Escalation thresholds — when does a human take over?
ESCALATE_IF = {
    "sentiment_negative_turns": 2,
    "tool_failures": 2,
    "turns": 10,
    "explicit_request": ["human", "agent", "manager", "शिकायत", "इंसान"],
}

def should_escalate(state):
    if state["turns"] >= ESCALATE_IF["turns"]: return "turn limit"
    if state["tool_failures"] >= ESCALATE_IF["tool_failures"]: return "repeated tool failure"
    if state["neg_turns"] >= ESCALATE_IF["sentiment_negative_turns"]: return "customer frustrated"
    if any(k in state["last_msg"].lower() for k in ESCALATE_IF["explicit_request"]):
        return "explicitly requested"
    return None

for s in [{"turns": 3, "tool_failures": 0, "neg_turns": 0, "last_msg": "what is my balance"},
          {"turns": 4, "tool_failures": 0, "neg_turns": 0, "last_msg": "मुझे इंसान से बात करनी है"},
          {"turns": 11, "tool_failures": 0, "neg_turns": 0, "last_msg": "ok"}]:
    print(f"{str(should_escalate(s) or 'continue'):<24} ← {s['last_msg']}")

continue                 ← what is my balance
explicitly requested     ← मुझे इंसान से बात करनी है
turn limit               ← ok


---
## 5 · The eval harness — the module that wins pilots

An enterprise buyer's first question is *"does it work"*. Their second, and the one
that decides the contract, is **"how do you know it still works next month?"**

In [12]:
# ── Golden set. Ten cases from REAL transcripts beats fifty invented ones. ──

ACCOUNTS = {"LN1001": {"name": "Rajesh Kumar", "emi": 12500, "due": "2026-08-15",
                       "outstanding": 340000}}

GOLDEN = [
    {"id": "g1", "input": "मेरे अकाउंट LN1001 की अगली EMI कब है?",
     "expect_tool": "get_emi_schedule", "expect_lang": "hi", "must_contain": ["15"]},
    {"id": "g2", "input": "What is the outstanding on LN1001?",
     "expect_tool": "get_account", "expect_lang": "en", "must_contain": ["340000", "3,40,000"]},
    {"id": "g3", "input": "LN1001 auto-debit failed, raise a complaint",
     "expect_tool": "raise_ticket", "expect_lang": "en", "must_contain": ["TKT"]},
    {"id": "g4", "input": "What is the capital of France?",
     "expect_tool": None, "expect_lang": "en", "must_contain": []},
    # g5 — hallucination check. LN9999 doesn't exist, so get_account returns
    # {"error": "not found"}. A model honouring "never invent numbers" in the 
    # ssytem prompt relays that; one that doesn't may fabricate a plausible-looking 
    # balance instead.
    {"id": "g5", "input": "What is the outstanding balance on account LN9999?",
     "expect_tool": "get_account", "expect_lang": "en",
     "must_contain": ["not found", "no record", "does not exist", "doesn't exist",
                       "couldn't find", "could not find", "unable to find", "no account"]},
]

def detect_lang(text):
    return "hi" if any("\u0900" <= ch <= "\u097F" for ch in text) else "en"

In [13]:
# Reuse the tool agent from Lab 05 (redefined compactly here)
# Reuse the tool agent from Lab 05 (redefined compactly here)

def get_account(account_id): return ACCOUNTS.get(account_id, {"error": "not found"})
def get_emi_schedule(account_id):
    a = ACCOUNTS.get(account_id)
    return {"error": "nf"} if not a else {"next_due": a["due"], "amount": a["emi"]}
def raise_ticket(account_id, issue): return {"ticket_id": "TKT-88214", "issue": issue}
REGISTRY = {"get_account": get_account, "get_emi_schedule": get_emi_schedule,
            "raise_ticket": raise_ticket}
TOOLS = [
  {"type":"function","function":{"name":"get_account","description":"Loan account details",
   "parameters":{"type":"object","properties":{"account_id":{"type":"string","description":"Account no"}},"required":["account_id"]}}},
  {"type":"function","function":{"name":"get_emi_schedule","description":"Upcoming EMI schedule",
   "parameters":{"type":"object","properties":{"account_id":{"type":"string","description":"Account no"}},"required":["account_id"]}}},
  {"type":"function","function":{"name":"raise_ticket","description":"Raise a support ticket",
   "parameters":{"type":"object","properties":{"account_id":{"type":"string","description":"Account no"},
    "issue":{"type":"string","description":"Problem"}},"required":["account_id","issue"]}}},
]

SYSTEM = ("You are a loan servicing agent. Use tools for any account fact. "
          "Never invent numbers. Reply in the user's language. "
          "If an account id does not match any record, say so plainly — "
          "never guess or invent account details.")

def agent(user_msg, seed=42):
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":user_msg}]
    used, calls_log, inr, t0 = [], [], 0.0, time.perf_counter()
    for _ in range(4):
        r = client.chat.completions(model="sarvam-105b", messages=msgs, tools=TOOLS,
                                    max_tokens=1500, seed=seed)
        inr += cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
        m = r.choices[0].message
        calls = getattr(m, "tool_calls", None)
        if not calls:
            return {"text": m.content or "", "tools": used, "calls": calls_log,
                    "ms": (time.perf_counter()-t0)*1000, "inr": inr}
        msgs.append({"role":"assistant","content":m.content,"tool_calls":
            [{"id":c.id,"type":"function","function":{"name":c.function.name,
              "arguments":c.function.arguments}} for c in calls]})
        for c in calls:
            used.append(c.function.name)
            calls_log.append(f"{c.function.name}({c.function.arguments})")
            out = REGISTRY[c.function.name](**json.loads(c.function.arguments))
            msgs.append({"role":"tool","tool_call_id":c.id,
                         "content":json.dumps(out, ensure_ascii=False)})
    return {"text":"max turns","tools":used,"calls":calls_log,
            "ms":(time.perf_counter()-t0)*1000,"inr":inr}

In [14]:
def run_evals(cases=GOLDEN, verbose=True):
    rows = []
    for c in cases:
        out = agent(c["input"])
        tool_ok = (c["expect_tool"] in out["tools"]) if c["expect_tool"] else (not out["tools"])
        lang_ok = detect_lang(out["text"]) == c["expect_lang"]
        cont_ok = (not c["must_contain"]) or any(
            k.replace(",", "") in out["text"].replace(",", "") for k in c["must_contain"])
        rows.append({"id": c["id"], "tool": tool_ok, "lang": lang_ok,
                     "content": cont_ok, "ms": out["ms"], "inr": out["inr"],
                     "pass": tool_ok and lang_ok and cont_ok, "text": out["text"][:60],
                     "calls": out["calls"]})
        if verbose:
            mark = "✅" if rows[-1]["pass"] else "❌"
            print(f"{mark} {c['id']}  tool={tool_ok!s:<5} lang={lang_ok!s:<5} "
                  f"content={cont_ok!s:<5} {out['ms']:>6.0f}ms ₹{out['inr']:.4f}")
            print(f"     call: {'; '.join(out['calls']) or '(no tool call — answered directly)'}")
    return rows

def summarise(rows):
    n = len(rows)
    lat = sorted(r["ms"] for r in rows)
    print("\n" + "═" * 52)
    print(f"pass rate        {sum(r['pass'] for r in rows)}/{n}  "
          f"({sum(r['pass'] for r in rows)/n:.0%})")
    print(f"tool accuracy    {sum(r['tool'] for r in rows)/n:.0%}")
    print(f"language correct {sum(r['lang'] for r in rows)/n:.0%}")
    print(f"latency p50      {lat[len(lat)//2]:.0f} ms")
    print(f"latency p95      {lat[int(len(lat)*0.95)-1]:.0f} ms")
    print(f"mean cost/task   ₹{sum(r['inr'] for r in rows)/n:.4f}")
    print("═" * 52)

baseline = run_evals(); summarise(baseline)

✅ g1  tool=True  lang=True  content=True    6135ms ₹0.0415
     call: get_emi_schedule({"account_id": "LN1001"})
✅ g2  tool=True  lang=True  content=True    3212ms ₹0.0342
     call: get_account({"account_id": "LN1001"})
✅ g3  tool=True  lang=True  content=True    7380ms ₹0.0495
     call: raise_ticket({"account_id": "LN1001", "issue": "Auto-debit failed"})
✅ g4  tool=True  lang=True  content=True    5484ms ₹0.0309
     call: (no tool call — answered directly)
✅ g5  tool=True  lang=True  content=True    3143ms ₹0.0324
     call: get_account({"account_id": "LN9999"})

════════════════════════════════════════════════════
pass rate        5/5  (100%)
tool accuracy    100%
language correct 100%
latency p50      5484 ms
latency p95      6135 ms
mean cost/task   ₹0.0377
════════════════════════════════════════════════════


In [15]:
# ── Now break a prompt deliberately and watch the harness catch it ─────────
SYSTEM = "You are a helpful assistant."      # ← tool discipline removed

broken = run_evals(); summarise(broken)

print("\nDIFFERENCES vs baseline (what changed, and what each side actually asked the tool):")
for b, k in zip(baseline, broken):
    if b["pass"] != k["pass"]:
        print(f"  {b['id']}: {b['pass']} → {k['pass']}   «{k['text']}»")
        print(f"      baseline call: {'; '.join(b['calls']) or '(no tool call)'}")
        print(f"      broken   call: {'; '.join(k['calls']) or '(no tool call)'}")

✅ g1  tool=True  lang=True  content=True    4360ms ₹0.0353
     call: get_emi_schedule({"account_id": "LN1001"})
✅ g2  tool=True  lang=True  content=True    4708ms ₹0.0334
     call: get_account({"account_id": "LN1001"})
✅ g3  tool=True  lang=True  content=True    4185ms ₹0.0349
     call: raise_ticket({"account_id": "LN1001", "issue": "Auto-debit failed"})
✅ g4  tool=True  lang=True  content=True    1756ms ₹0.0131
     call: (no tool call — answered directly)
❌ g5  tool=True  lang=True  content=False   4795ms ₹0.0304
     call: get_account({"account_id": "LN9999"})

════════════════════════════════════════════════════
pass rate        4/5  (80%)
tool accuracy    100%
language correct 100%
latency p50      4360 ms
latency p95      4708 ms
mean cost/task   ₹0.0294
════════════════════════════════════════════════════

DIFFERENCES vs baseline (what changed, and what each side actually asked the tool):
  g5: True → False   «I wasn't able to find an account with the ID **LN9999** in o»
    

> **A single 5-case, non-deterministic run can't tell you whether a prompt change
> helped, hurt, or did nothing.** One flipped case is a 20-point swing, and `agent()`
> isn't called at `temperature=0` — baseline and broken are two separate rolls of the
> dice, not a controlled comparison. Don't read a directional story into either score
> without repeating the run.
>
> What the run *does* show is still useful: even a bare `"You are a helpful
> assistant."` gets tool selection right most of the time — that's the tool schema
> doing the work, not the system prompt. The one thing schema alone can't cover is
> what to do when a tool comes back **empty**. `g5` (a non-existent account) is
> exactly that gap — close it with an explicit instruction rather than leaving it to
> the model's judgement: *"if an account id doesn't match any record, say so plainly
> — never guess or invent details."* That's now baked into `SYSTEM` above.
>
> To see *why* two runs disagree, not just *that* they disagree, print what each
> tool call was actually asked — the argument, not just the name (see the `calls`
> field above). If one run's `get_account` was called with the right id and the model
> answered from its result, while the other run answered directly without checking,
> both can be individually reasonable — the disagreement is about *whether to check*,
> not about a broken tool. That's a more specific, more actionable signal than "it
> regressed."

In [16]:
SYSTEM = ("You are a loan servicing agent. Use tools for any account fact. "
          "Never invent numbers. Reply in the user's language. "
          "If an account id does not match any record, say so plainly — "
          "never guess or invent account details.")     # restore
cost.report()

LLM          ₹   0.1841  37 in / 2500 out
LLM          ₹   0.0197  22 in / 260 out
LLM          ₹   0.0157  335 in / 81 out
LLM          ₹   0.0258  401 in / 192 out
LLM          ₹   0.0143  333 in / 62 out
LLM          ₹   0.0200  414 in / 107 out
LLM          ₹   0.0197  336 in / 135 out
LLM          ₹   0.0298  410 in / 243 out
LLM          ₹   0.0309  329 in / 290 out
LLM          ₹   0.0138  335 in / 54 out
LLM          ₹   0.0186  379 in / 103 out
LLM          ₹   0.0146  294 in / 82 out
LLM          ₹   0.0207  360 in / 139 out
LLM          ₹   0.0176  292 in / 123 out
LLM          ₹   0.0158  373 in / 67 out
LLM          ₹   0.0165  295 in / 107 out
LLM          ₹   0.0184  369 in / 104 out
LLM          ₹   0.0131  288 in / 64 out
LLM          ₹   0.0131  294 in / 62 out
LLM          ₹   0.0173  338 in / 101 out
TOTAL        ₹   0.5395
              (₹1000 free credit → ₹999.46 left)
              Estimated from published rates; actual billing usually lower


0.5394547199999999

---
## ✅ Checkpoint

- [ ] You moved a calculation out of the model and into code
- [ ] A workflow failed at step 3, resumed from checkpoint, and did not re-run steps 0–2
- [ ] Tracer output shows per-span latency, tokens and ₹
- [ ] PII is redacted before anything is logged
- [ ] The eval harness caught a deliberate prompt regression

## 🧪 Try this

1. Grow the golden set to 10 cases **from your own domain**.
2. Add a `hallucination` check: ask about an account that does not exist. Does it invent?
3. Add retry-with-backoff around the tool calls and inject a flaky tool.
4. Persist checkpoints to a real database and resume across a kernel restart.
5. Add cost-per-task to your escalation rules — escalate when a conversation exceeds ₹X.